

# 🙏 Credits

This notebook is based on and inspired by:

- **Nathalia Chaves**  
  🔗 [EV Blend ⚡](https://www.kaggle.com/code/nathaliach/ev-blend/notebook) *(Primary inspiration for this notebook)*

- **Chinzorigt Ganbat**  
  🔗 [S6E9: Does Breaking Ties Help?](https://www.kaggle.com/code/chinzorigtganbat/s6e9-does-breaking-ties-help)

- **Naji Ama**  
  🔗 [XGBoost \| Triple-TE & Dynamic Pruning (LB 0.94639)](https://www.kaggle.com/code/najiama/xgboost-triple-te-dynamic-pruning-lb-0-94639)

- **Talha Tursun**  
  🔗 [S6E9 \| Daily Rank-Average Ensemble](https://www.kaggle.com/code/talhatursun/s6e9-daily-rank-average-ensemble)

> **Acknowledgement:** This notebook is intended for **learning and educational purposes**. Full credit goes to the original authors for sharing their ideas and notebooks with the Kaggle community.

#  Rank Blending Explained

This notebook does not train any model. Instead it combines predictions from existing models to create better solution. I Used technique called Blending which is a popular ensemble method.

##  Workflow

```text
Prediction File 1 (Anchor)
          │
          ▼
    Convert to Ranks
          │
          │
Prediction File 2 (LGBM)
          │
          ▼
    Convert to Ranks
          │
          ▼
Weighted Blend
(0.80 × Anchor - 0.20 × Reg1)
          │
          ▼
   Rank Again
          │
          ▼
 submission.csv
```

---

##  What is Rank Blending?

Instead of combining prediction probabilities, We are combining their ranks (positions).

Example:

```text
Probabilities:
0.95
0.72
0.20

↓

Ranks:
3
2
1
```

Only the order of predictions matters, not the exact values.

---

##  Why use Ranks?

 Because different models produce probabilities on different scales.

Example:

```text
Model A: 0.99, 0.95, 0.10
Model B: 0.70, 0.65, 0.30
```

Both models agree on the order of samples, but not on the probability values.

Converting to ranks makes them comparable and often gives a more stable ensemble.

---

##  Key Points

- No model training is done.
- Predictions from two models are combined.
- Predictions are converted to ranks before blending.
- Rank blending is more robust than averaging probabilities.
- The Anchor model is treated as the strongest model.
- Final predictions are saved as submission.csv.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata



PREDICTION_FILES = {
    "anchor": (
        "/kaggle/input/notebooks/talhatursun/"
        "s6e9-daily-rank-average-ensemble/submission.csv"
    ),
    "lgbm": (
        "/kaggle/input/datasets/najiama/"
        "s6e9-oof/Pure LGBM_V3_test.csv"
    ),
}

TARGET = "Will_Buy_EV"

WEIGHTS = {
    "anchor": 0.80,
    "lgbm": 0.20,
}



def normalized_rank(values: np.ndarray) -> np.ndarray:
    """Convert predictions to the interval (0, 1) using ranks."""
    return rankdata(values, method="average") / (len(values) + 1)


def load_prediction(path: str) -> np.ndarray:
    df = pd.read_csv(path)
    return df[TARGET].to_numpy()



ranked_predictions = {}

for name, path in PREDICTION_FILES.items():
    preds = load_prediction(path)
    ranked_predictions[name] = normalized_rank(preds)


ensemble = np.zeros_like(next(iter(ranked_predictions.values())))

for name, ranks in ranked_predictions.items():
    ensemble += WEIGHTS[name] * ranks

ensemble = normalized_rank(ensemble)



submission = pd.read_csv(
    "/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv"
)

submission[TARGET] = ensemble
submission.to_csv("submission.csv", index=False)

submission.head()

---

If this notebook helped you understand Rank Blending, ensemble techniques, or gave you a new idea, please consider upvoting  it.

**Happy Kaggle-ing!**